In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier   
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report

def rfeFeature(indep_X,dep_Y,n):
        rfelist=[]
        
        log_model = LogisticRegression(solver='lbfgs')
        RF = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
       # NB = GaussianNB()
        DT= DecisionTreeClassifier(criterion = 'gini', max_features='sqrt',splitter='best',random_state = 0)
        svc_model = SVC(kernel = 'linear', random_state = 0)
        #knn = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        rfemodellist=[log_model,svc_model,RF,DT] 
        for i in   rfemodellist:
            print(i)
            log_rfe = RFE(i, n)
            log_fit = log_rfe.fit(indep_X, dep_Y)
            log_rfe_feature=log_fit.transform(indep_X)
            rfelist.append(log_rfe_feature)
        return rfelist

In [3]:
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        #X_train, X_test, y_train, y_test = train_test_split(indep_X,dep_Y, test_size = 0.25, random_state = 0)
        
        #Feature Scaling
        #from sklearn.preprocessing import StandardScaler
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)
        
        return X_train, X_test, y_train, y_test

In [4]:
def cm_prediction(classifier,X_test):
    y_pred = classifier.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    Accuracy=accuracy_score(y_test, y_pred )
    report=classification_report(y_test, y_pred)
    return  classifier,Accuracy,report,X_test,y_test,cm

In [5]:
def logistic(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LogisticRegression
        classifier = LogisticRegression(random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      
    
def svm_linear(X_train,y_train,X_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'linear', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
    
def svm_NL(X_train,y_train,X_test):
                
        from sklearn.svm import SVC
        classifier = SVC(kernel = 'rbf', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
def Navie(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.naive_bayes import GaussianNB
        classifier = GaussianNB()
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm         
    
    
def knn(X_train,y_train,X_test):
           
        # Fitting K-NN to the Training set
        from sklearn.neighbors import KNeighborsClassifier
        classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p = 2)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm
def Decision(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training set
        from sklearn.tree import DecisionTreeClassifier
        classifier = DecisionTreeClassifier(criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm      


def random(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestClassifier
        classifier = RandomForestClassifier(n_estimators = 10, criterion = 'entropy', random_state = 0)
        classifier.fit(X_train, y_train)
        classifier,Accuracy,report,X_test,y_test,cm=cm_prediction(classifier,X_test)
        return  classifier,Accuracy,report,X_test,y_test,cm

In [6]:
def rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf): 
    
    rfedataframe=pd.DataFrame(index=['Logistic','SVC','Random','DecisionTree'],columns=['Logistic','SVMl','SVMnl',
                                                                                        'KNN','Navie','Decision','Random'])

    for number,idex in enumerate(rfedataframe.index):
        
        rfedataframe['Logistic'][idex]=acclog[number]       
        rfedataframe['SVMl'][idex]=accsvml[number]
        rfedataframe['SVMnl'][idex]=accsvmnl[number]
        rfedataframe['KNN'][idex]=accknn[number]
        rfedataframe['Navie'][idex]=accnav[number]
        rfedataframe['Decision'][idex]=accdes[number]
        rfedataframe['Random'][idex]=accrf[number]
    return rfedataframe

In [7]:
dataset1=pd.read_csv("train_preprocessed.csv",index_col=None)
df2=dataset1
df2 = pd.get_dummies(df2, drop_first=True)

indep_X=df2.drop(columns=['PassengerId','Survived'])
dep_Y=df2['Survived']

In [8]:
def rfeFeature(indep_X, dep_Y, n):
    rfelist = []
    selected_features = {}

    log_model = LogisticRegression(solver='lbfgs', max_iter=200)
    RF = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)
    DT = DecisionTreeClassifier(criterion='gini', max_features='sqrt', splitter='best', random_state=0)
    svc_model = SVC(kernel='linear', random_state=0)

    rfemodellist = {
        "LogisticRegression": log_model,
        "SVC": svc_model,
        "RandomForest": RF,
        "DecisionTree": DT
    }

    for name, model in rfemodellist.items():
        print(f"\nModel: {name}")
        # ✅ fix: explicitly name the argument
        rfe = RFE(estimator=model, n_features_to_select=n)
        fit = rfe.fit(indep_X, dep_Y)

        # Transform dataset
        X_rfe = fit.transform(indep_X)
        rfelist.append(X_rfe)

        # Get selected features
        selected = indep_X.columns[fit.support_]
        selected_features[name] = list(selected)
        print(f"Selected features: {list(selected)}")

    return rfelist, selected_features


In [10]:
rfelist, selected_features = rfeFeature(indep_X, dep_Y, 4)

print("\nSummary of selected features by each model:")
for model, feats in selected_features.items():
    print(f"{model}: {feats}")


Model: LogisticRegression
Selected features: ['Pclass', 'SibSp', 'Sex_male', 'Embarked_S']

Model: SVC
Selected features: ['Pclass', 'SibSp', 'Sex_male', 'Embarked_S']

Model: RandomForest
Selected features: ['Pclass', 'Age', 'Fare', 'Sex_male']

Model: DecisionTree
Selected features: ['Pclass', 'Age', 'Fare', 'Sex_male']

Summary of selected features by each model:
LogisticRegression: ['Pclass', 'SibSp', 'Sex_male', 'Embarked_S']
SVC: ['Pclass', 'SibSp', 'Sex_male', 'Embarked_S']
RandomForest: ['Pclass', 'Age', 'Fare', 'Sex_male']
DecisionTree: ['Pclass', 'Age', 'Fare', 'Sex_male']


In [11]:
rfelist

[array([[3., 1., 1., 1.],
        [1., 1., 0., 0.],
        [3., 0., 0., 1.],
        ...,
        [3., 1., 0., 1.],
        [1., 0., 1., 0.],
        [3., 0., 1., 0.]], shape=(891, 4)),
 array([[3., 1., 1., 1.],
        [1., 1., 0., 0.],
        [3., 0., 0., 1.],
        ...,
        [3., 1., 0., 1.],
        [1., 0., 1., 0.],
        [3., 0., 1., 0.]], shape=(891, 4)),
 array([[ 3.    , 22.    ,  7.25  ,  1.    ],
        [ 1.    , 38.    , 71.2833,  0.    ],
        [ 3.    , 26.    ,  7.925 ,  0.    ],
        ...,
        [ 3.    , 28.    , 23.45  ,  0.    ],
        [ 1.    , 26.    , 30.    ,  1.    ],
        [ 3.    , 32.    ,  7.75  ,  1.    ]], shape=(891, 4)),
 array([[ 3.    , 22.    ,  7.25  ,  1.    ],
        [ 1.    , 38.    , 71.2833,  0.    ],
        [ 3.    , 26.    ,  7.925 ,  0.    ],
        ...,
        [ 3.    , 28.    , 23.45  ,  0.    ],
        [ 1.    , 26.    , 30.    ,  1.    ],
        [ 3.    , 32.    ,  7.75  ,  1.    ]], shape=(891, 4))]

In [12]:
acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

for i in rfelist:   
    X_train, X_test, y_train, y_test=split_scalar(i,dep_Y)   
    
        
    classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test)
    acclog.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_linear(X_train,y_train,X_test)  
    accsvml.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_NL(X_train,y_train,X_test)  
    accsvmnl.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=knn(X_train,y_train,X_test)  
    accknn.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=Navie(X_train,y_train,X_test)  
    accnav.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=Decision(X_train,y_train,X_test)  
    accdes.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test)  
    accrf.append(Accuracy)
    
result=rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)

result

C:\Users\sathi\AppData\Local\Temp\ipykernel_19836\2386145136.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  rfedataframe['Logistic'][idex]=acclog[number]
C:\Users\sathi\AppData\Local\Temp\ipykernel_19836\2386145136.py:9: FutureWarning: 

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
Logistic,0.789238,0.780269,0.811659,0.789238,0.789238,0.793722,0.793722
SVC,0.789238,0.780269,0.811659,0.789238,0.789238,0.793722,0.793722
Random,0.793722,0.780269,0.798206,0.820628,0.775785,0.784753,0.843049
DecisionTree,0.793722,0.780269,0.798206,0.820628,0.775785,0.784753,0.843049


In [19]:
#from the observation RFE also selects Pclass,Age,Fare,Sex_male as top 4 features for prediction and both Decision Tree and Random Forest gives 84% Accuracy

In [15]:
rfelist, selected_features = rfeFeature(indep_X, dep_Y, 5)

print("\nSummary of selected features by each model:")
for model, feats in selected_features.items():
    print(f"{model}: {feats}")


Model: LogisticRegression
Selected features: ['Pclass', 'SibSp', 'Sex_male', 'Embarked_Q', 'Embarked_S']

Model: SVC
Selected features: ['Pclass', 'SibSp', 'Parch', 'Sex_male', 'Embarked_S']

Model: RandomForest
Selected features: ['Pclass', 'Age', 'SibSp', 'Fare', 'Sex_male']

Model: DecisionTree
Selected features: ['Pclass', 'Age', 'SibSp', 'Fare', 'Sex_male']

Summary of selected features by each model:
LogisticRegression: ['Pclass', 'SibSp', 'Sex_male', 'Embarked_Q', 'Embarked_S']
SVC: ['Pclass', 'SibSp', 'Parch', 'Sex_male', 'Embarked_S']
RandomForest: ['Pclass', 'Age', 'SibSp', 'Fare', 'Sex_male']
DecisionTree: ['Pclass', 'Age', 'SibSp', 'Fare', 'Sex_male']


In [16]:
acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

for i in rfelist:   
    X_train, X_test, y_train, y_test=split_scalar(i,dep_Y)   
    
        
    classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test)
    acclog.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_linear(X_train,y_train,X_test)  
    accsvml.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_NL(X_train,y_train,X_test)  
    accsvmnl.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=knn(X_train,y_train,X_test)  
    accknn.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=Navie(X_train,y_train,X_test)  
    accnav.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=Decision(X_train,y_train,X_test)  
    accdes.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test)  
    accrf.append(Accuracy)
    
result=rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)

result

C:\Users\sathi\AppData\Local\Temp\ipykernel_19836\2386145136.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  rfedataframe['Logistic'][idex]=acclog[number]
C:\Users\sathi\AppData\Local\Temp\ipykernel_19836\2386145136.py:9: FutureWarning: 

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
Logistic,0.793722,0.780269,0.811659,0.793722,0.789238,0.793722,0.798206
SVC,0.789238,0.780269,0.789238,0.798206,0.789238,0.807175,0.766816
Random,0.789238,0.780269,0.802691,0.811659,0.780269,0.793722,0.816143
DecisionTree,0.789238,0.780269,0.802691,0.811659,0.780269,0.793722,0.816143


In [17]:
rfelist, selected_features = rfeFeature(indep_X, dep_Y, 3)

print("\nSummary of selected features by each model:")
for model, feats in selected_features.items():
    print(f"{model}: {feats}")


Model: LogisticRegression
Selected features: ['Pclass', 'Sex_male', 'Embarked_S']

Model: SVC
Selected features: ['Pclass', 'SibSp', 'Sex_male']

Model: RandomForest
Selected features: ['Age', 'Fare', 'Sex_male']

Model: DecisionTree
Selected features: ['Age', 'Fare', 'Sex_male']

Summary of selected features by each model:
LogisticRegression: ['Pclass', 'Sex_male', 'Embarked_S']
SVC: ['Pclass', 'SibSp', 'Sex_male']
RandomForest: ['Age', 'Fare', 'Sex_male']
DecisionTree: ['Age', 'Fare', 'Sex_male']


In [18]:
acclog=[]
accsvml=[]
accsvmnl=[]
accknn=[]
accnav=[]
accdes=[]
accrf=[]

for i in rfelist:   
    X_train, X_test, y_train, y_test=split_scalar(i,dep_Y)   
    
        
    classifier,Accuracy,report,X_test,y_test,cm=logistic(X_train,y_train,X_test)
    acclog.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_linear(X_train,y_train,X_test)  
    accsvml.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=svm_NL(X_train,y_train,X_test)  
    accsvmnl.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=knn(X_train,y_train,X_test)  
    accknn.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=Navie(X_train,y_train,X_test)  
    accnav.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=Decision(X_train,y_train,X_test)  
    accdes.append(Accuracy)
    
    classifier,Accuracy,report,X_test,y_test,cm=random(X_train,y_train,X_test)  
    accrf.append(Accuracy)
    
result=rfe_classification(acclog,accsvml,accsvmnl,accknn,accnav,accdes,accrf)

result

C:\Users\sathi\AppData\Local\Temp\ipykernel_19836\2386145136.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  rfedataframe['Logistic'][idex]=acclog[number]
C:\Users\sathi\AppData\Local\Temp\ipykernel_19836\2386145136.py:9: FutureWarning: 

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
Logistic,0.780269,0.780269,0.811659,0.766816,0.784753,0.811659,0.811659
SVC,0.789238,0.780269,0.784753,0.798206,0.7713,0.793722,0.789238
Random,0.775785,0.780269,0.789238,0.784753,0.775785,0.7713,0.807175
DecisionTree,0.775785,0.780269,0.789238,0.784753,0.775785,0.7713,0.807175


In [ ]:
#from the observation RFE also selects Pclass,Age,Fare,Sex_male as top 4 features for prediction and both Decision Tree and Random Forest gives 84% Accuracy